[Python, Visually](https://johnfisher-ai.github.io/Python-Visual-Guides/) &nbsp;&rsaquo;&nbsp; [PyTorch and TensorFlow, Deep Dive](https://johnfisher-ai.github.io/Python-Visual-Guides/pytorch-and-tensorflow-deep-dive.html)

# Why PyTorch and TensorFlow &middot; Solutions


One way to do each task. Not the only way. If yours runs and does what was asked, yours is
right too.

In [1]:
import contextlib
import os

import numpy as np
import torch


@contextlib.contextmanager
def c_messages_hidden():
    """Send what C and C++ code writes to the error stream to os.devnull while the block runs."""
    saved = os.dup(2)
    devnull = os.open(os.devnull, os.O_WRONLY)
    os.dup2(devnull, 2)
    try:
        yield
    finally:
        os.dup2(saved, 2)
        os.close(devnull)
        os.close(saved)


os.environ["TF_CPP_MIN_LOG_LEVEL"] = "3"
with c_messages_hidden():
    import tensorflow as tf
    import keras


def station_network_arrays(n=1200, seed=2025):
    """The made-up stations of the scikit-learn guide, as arrays: four features and two targets."""
    rng = np.random.default_rng(seed)
    latitude = rng.uniform(58.0, 71.0, n)
    longitude = 5 + 1.5 * (latitude - 58) + rng.uniform(0, 5, n)
    coast_km = np.minimum(rng.exponential(25, n), 250)
    elevation = np.clip(rng.gamma(1.5, 150, n) + 3 * coast_km, 0, 1800)
    shift = np.where(coast_km < 3, 0.8,
            np.where(elevation > 700, -0.5,
            np.where((coast_km < 30) & (rng.uniform(size=n) < 0.6), 0.3, 0.0)))
    rng.choice(4, n, p=[0.4, 0.3, 0.1, 0.2])   # each station's owner, which this guide leaves out
    rng.integers(1950, 2025, n)                 # and the year it was installed
    jan_mean = (1.5 - 0.55 * (latitude - 58) - 0.006 * elevation - 4.0 * (1 - np.exp(-coast_km / 40))
                + shift + rng.normal(0, 0.9, n))
    may_mean = jan_mean + 8.5 + 0.004 * coast_km + rng.normal(0, 1.2, n)
    features = np.column_stack([latitude.round(2), longitude.round(2), elevation.round(0), coast_km.round(1)])
    return features.astype("float32"), jan_mean.round(1).astype("float32"), (may_mean < 2.5).astype("float32")


FEATURES = ["latitude", "longitude", "elevation_m", "coast_km"]
features, jan_mean, snow_on_may_1 = station_network_arrays()
order = np.random.default_rng(0).permutation(len(features))
train_rows, valid_rows, test_rows = order[:800], order[800:1000], order[1000:]
mean, std = features[train_rows].mean(axis=0), features[train_rows].std(axis=0)
scaled = (features - mean) / std

print("torch", torch.__version__.split("+")[0], "| tensorflow", tf.__version__, "| keras", keras.__version__,
      "|", len(features), "stations")

torch 2.11.0 | tensorflow 2.21.0 | keras 3.13.2 | 1200 stations


The worked examples' tensors, rebuilt here so that each answer runs on its own.

In [2]:
x = scaled[train_rows, 0]
y = jan_mean[train_rows]
X_train = torch.from_numpy(scaled[train_rows])
X_test = torch.from_numpy(scaled[test_rows])
y_train = torch.from_numpy(jan_mean[train_rows]).unsqueeze(1)
y_test = torch.from_numpy(jan_mean[test_rows]).unsqueeze(1)
loss_fn = torch.nn.MSELoss()

**1.** Run the NumPy line fit again with a learning rate of 0.01 in place of 0.1, and print the loss every 25 steps to see whether 100 steps is enough.

In [3]:
w, b = 0.0, 0.0
for step in range(101):
    error = w * x + b - y
    loss = (error ** 2).mean()
    if step % 25 == 0:
        print(step, round(float(loss), 3))
    w -= 0.01 * 2 * (error * x).mean()
    b -= 0.01 * 2 * error.mean()

0 34.569
25 15.477
50 8.524
75 5.992
100 5.07


With a tenth of the step, 100 steps is not enough. The loss is still falling at step 100, at 5.07, where a learning rate of 0.1 had reached 4.542 by step 25.

**2.** Fit the January mean to elevation alone, column 2 of `scaled`, with the PyTorch loop, and print the slope in degrees per meter, the slope divided by `std[2]`.

In [4]:
x_elevation = torch.from_numpy(scaled[train_rows, 2])
y_torch = torch.from_numpy(y)
w = torch.tensor(0.0, requires_grad=True)
b = torch.tensor(0.0, requires_grad=True)
for step in range(101):
    loss = ((w * x_elevation + b - y_torch) ** 2).mean()
    loss.backward()
    with torch.no_grad():
        w -= 0.1 * w.grad
        b -= 0.1 * b.grad
    w.grad.zero_()
    b.grad.zero_()
print("loss", round(loss.item(), 3), "| degrees per meter", round(w.item() / float(std[2]), 4))

loss 6.146 | degrees per meter -0.0078


The January mean falls about 0.0078 degrees for every meter up, or 0.78 degrees per hundred meters. Elevation alone leaves a loss of 6.146, more than latitude's 4.542.

**3.** Write the four-feature fit in TensorFlow, with `tf.Variable(tf.zeros(4))` for the weights and `tf.linalg.matvec` for the product, and print the loss on the test stations after 500 steps.

In [5]:
X_train_tf = tf.constant(scaled[train_rows])
X_test_tf = tf.constant(scaled[test_rows])
y_train_tf = tf.constant(jan_mean[train_rows])
y_test_tf = tf.constant(jan_mean[test_rows])

w = tf.Variable(tf.zeros(4))
b = tf.Variable(0.0)
for step in range(500):
    with tf.GradientTape() as tape:
        loss = tf.reduce_mean((tf.linalg.matvec(X_train_tf, w) + b - y_train_tf) ** 2)
    dw, db = tape.gradient(loss, [w, b])
    w.assign_sub(0.1 * dw)
    b.assign_sub(0.1 * db)

test_loss = tf.reduce_mean((tf.linalg.matvec(X_test_tf, w) + b - y_test_tf) ** 2)
print("training loss", round(float(loss), 3), "| test loss", round(float(test_loss), 3))

training loss 0.992 | test loss 1.181


The same losses as the PyTorch version, 0.992 and 1.181. `tf.linalg.matvec` multiplies each row of the matrix by the vector of weights, as `@` does in PyTorch.

**4.** Train the PyTorch network with 64 hidden units in place of 32, from `torch.manual_seed(0)`, and print its loss on the test stations after 500 steps.

In [6]:
torch.manual_seed(0)
wide = torch.nn.Sequential(torch.nn.Linear(4, 64), torch.nn.ReLU(), torch.nn.Linear(64, 1))
optimizer = torch.optim.Adam(wide.parameters(), lr=0.01)
for step in range(500):
    optimizer.zero_grad()
    loss = loss_fn(wide(X_train), y_train)
    loss.backward()
    optimizer.step()
with torch.no_grad():
    print("training loss", round(loss.item(), 3), "| test loss", round(loss_fn(wide(X_test), y_test).item(), 3))

training loss 0.757 | test loss 0.984


The wider network fits its training stations better, 0.757 against 0.782, and does slightly worse on the test stations, 0.984 against 0.96. More hidden units let a network fit the training data more closely, which the **Overfitting and Regularization** notebook takes up.

**5.** Predict the January mean at a planned station at latitude 70.0, longitude 23.0, 300 meters up and 15 kilometers from the coast, with `net` and with `net_keras`.

In [7]:
torch.manual_seed(0)
net = torch.nn.Sequential(torch.nn.Linear(4, 32), torch.nn.ReLU(), torch.nn.Linear(32, 1))
optimizer = torch.optim.Adam(net.parameters(), lr=0.01)
for step in range(500):
    optimizer.zero_grad()
    loss = loss_fn(net(X_train), y_train)
    loss.backward()
    optimizer.step()

keras.utils.set_random_seed(0)
net_keras = keras.Sequential([keras.Input((4,)), keras.layers.Dense(32, activation="relu"), keras.layers.Dense(1)])
net_keras.compile(optimizer=keras.optimizers.Adam(0.01), loss="mse")
net_keras.fit(scaled[train_rows], jan_mean[train_rows], epochs=500, batch_size=800, verbose=0)

planned = ((np.array([[70.0, 23.0, 300.0, 15.0]]) - mean) / std).astype("float32")
with torch.no_grad():
    print("PyTorch:", round(net(torch.from_numpy(planned)).item(), 1))
print("Keras:", round(float(net_keras.predict(planned, verbose=0)[0, 0]), 1))

PyTorch: -7.9
Keras: -7.9


Both models predict −7.9 degrees. The notebook's `net` and `net_keras` are trained again here from the same seeds, so the answers are the ones those two models give. `predict` gets `verbose=0` for the same reason as `fit`, since without it `predict` prints a progress bar, which the **Training Loops** notebook shows.

---

&#8592; **Back to:** [Why PyTorch and TensorFlow](https://colab.research.google.com/github/johnfisher-ai/Python-Visual-Guides/blob/main/notebooks/pytorch-and-tensorflow-deep-dive/01-why-pytorch-and-tensorflow.ipynb)  &nbsp;&middot;&nbsp;  [PyTorch and TensorFlow, Deep Dive Notebooks](https://johnfisher-ai.github.io/Python-Visual-Guides/pytorch-and-tensorflow-deep-dive.html)
